# Markov Chain Poetry Generator

## 1. File Reading

Read the poem from `robert_frost.txt` line by line, remove empty lines, and handle file errors.

In [ ]:
file_name = "robert_frost.txt"

try:
  f = open(file_name, "r", encoding="utf-8")
  lines = f.readlines()
  print(lines[:5])
except FileNotFoundError:
  print(f"Error: file '{file_name}' not found!")

['Two roads diverged in a yellow wood,\n', 'And sorry I could not travel both\n', 'And be one traveler, long I stood\n', 'And looked down one as far as I could\n', 'To where it bent in the undergrowth; \n']


In [ ]:
clean_lines = []
for x in lines:
  x = x.strip()
  if x != "":
    clean_lines.append(x)

print("Lines before", len(lines))
print("Lines after", len(clean_lines))
print(clean_lines[:5])

Lines before 1581
Lines after 1436
['Two roads diverged in a yellow wood,', 'And sorry I could not travel both', 'And be one traveler, long I stood', 'And looked down one as far as I could', 'To where it bent in the undergrowth;']


## 2. Text Processing
Remove punctuation, convert to lowercase, and split each line into words.

In [ ]:
import re

tokenized_lines = []
for t in clean_lines:
  t = t.lower()
  t = re.sub(r'[^a-zA-Z\s]', '', t)
  t = t.split()
  tokenized_lines.append(t)

print(tokenized_lines[:5])

[['two', 'roads', 'diverged', 'in', 'a', 'yellow', 'wood'], ['and', 'sorry', 'i', 'could', 'not', 'travel', 'both'], ['and', 'be', 'one', 'traveler', 'long', 'i', 'stood'], ['and', 'looked', 'down', 'one', 'as', 'far', 'as', 'i', 'could'], ['to', 'where', 'it', 'bent', 'in', 'the', 'undergrowth']]


## 3. Data Structures
Build three dictionaries: initial word probabilities, first-order transitions, and second-order transitions.

### A. Initial Word Probabilities
Probability of each word appearing at the start of a line.

In [ ]:
initial_counts = {}

for line in tokenized_lines:
  first_word = line[0]

  if first_word in initial_counts:
    initial_counts[first_word] = initial_counts[first_word] + 1
  else:
    initial_counts[first_word] = 1

print(len(initial_counts)) # Number of different first words

305


In [ ]:
initial_probs = {}

for word in initial_counts:
  initial_probs[word] = initial_counts[word] / len(tokenized_lines)

print(sum(initial_probs.values()))

1.0


### B. First-order Transitions
Probability of a word following another word.

In [ ]:
first_order_counts = {}

for line in tokenized_lines:
  words = line + ["END"]
  for i in range(len(words) - 1):
    prev = words[i]
    nxt = words[i + 1]

    if prev not in first_order_counts:
      first_order_counts[prev] = {}

    if nxt in first_order_counts[prev]:
      first_order_counts[prev][nxt] = first_order_counts[prev][nxt] + 1
    else:
      first_order_counts[prev][nxt] = 1

print(len(first_order_counts)) # Number of different previous words

2190


In [ ]:
first_order_probs = {}

for prev in first_order_counts:
  total = sum(first_order_counts[prev].values())
  first_order_probs[prev] = {}
  for nxt in first_order_counts[prev]:
    first_order_probs[prev][nxt] = first_order_counts[prev][nxt] / total

print(first_order_probs["yellow"]) # Probabilities of words after "yellow"

{'wood': 0.5, 'strands': 0.5}


### C. Second-order Transitions
Probability of a word following two specific words.

In [ ]:
second_order_counts = {}

for line in tokenized_lines:
  words = line + ["END"]
  for i in range(len(words) - 2):
    prev2 = words[i]
    prev1 = words[i + 1]
    nxt = words[i + 2]
    key = (prev2, prev1)

    if key not in second_order_counts:
      second_order_counts[key] = {}

    if nxt in second_order_counts[key]:
      second_order_counts[key][nxt] = second_order_counts[key][nxt] + 1
    else:
      second_order_counts[key][nxt] = 1

print(len(second_order_counts))                 # Number of different word pairs

7031


In [ ]:
second_order_probs = {}

for key in second_order_counts:
  total = sum(second_order_counts[key].values())
  second_order_probs[key] = {}
  for nxt in second_order_counts[key]:
    second_order_probs[key][nxt] = second_order_counts[key][nxt] / total

print(second_order_probs[("miles", "to")])   # Probabilities after "miles to"

{'go': 1.0}


## 4. Word Selection
- Cumulative method: random selection based on probabilities
- Deterministic method: select the word with the highest probability

Cumulative method

In [ ]:
import random

def select_cumulative(probs):
  r = random.random()
  cumulative = 0
  for word in probs:
    cumulative = cumulative + probs[word]
    if r < cumulative:
      return word
  return word

print(select_cumulative(first_order_probs["yellow"]))   # Example: pick a word after "yellow"

wood


Deterministic method

In [ ]:
def select_deterministic(probs):
  return max(probs, key=probs.get)

print(select_deterministic(initial_probs))   # Example: most likely first word

and


## 5. Poetry Generation
Generate 4 lines: first word from initial probabilities, second from first-order, the rest from second-order transitions until 'END'.

In [ ]:
def generate_line(select_word):
  w1 = select_word(initial_probs)
  w2 = select_word(first_order_probs[w1])

  line = [w1, w2]

  while line[-1] != "END" and len(line) < 15:
    key = (line[-2], line[-1])
    nxt = select_word(second_order_probs[key])
    line.append(nxt)

  if line[-1] == "END":
    line = line[:-1]

  return " ".join(line)

print("Cumulative method:")
for i in range(4):
  print(generate_line(select_cumulative))

print()
print("Deterministic method:")
for i in range(4):
  print(generate_line(select_deterministic))

Cumulative method:
nothings too good for them i say
a gentle lot we are
beyond where they were crooking many a trillium
to watch me and take hold of me

Deterministic method:
and i
and i
and i
and i
